# Capstone starter: a support desk agent with its evidence

Booster LEAP, Agentic AI, topic 10. One small agent that does what the capstone asks: look up orders, search the policy, issue refunds only after a human approval, remember and forget a customer's contact preference, and refuse what is out of scope. Around it: tool contracts with validation (topic 2), stop rules (topic 3), memory with a purge (topic 4), a pause that waits for approval (topic 6), a policy layer and redaction (topic 8), a trace and an eval with a release gate (topic 9).

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). There is one TODO cell. The eval fails two tasks until you write it; then run the cells below it again and the release gate opens.

## Stage 1: the whole system with a stand-in model

The only **stand-in** is the model: a small class that follows a prepared plan of tool calls and a final answer. Everything around it is real code: the tool contracts and the validator, the policy layer, the loop with a step limit and a repeated-call check, the pause for approval, the trace with redaction, and the eval that checks the end state (the refunds issued and the memory), not the wording. The orders are a small stand-in for a real order service.

The setup cell: the support desk data (orders with owners, a policy text), the three tools (`lookup_order`, `search_policy`, `issue_refund`) and the registry with a contract per tool, which says whether it changes something. It prints nothing.

In [ ]:
import copy
import json
import re

CUSTOMER = "C1"                        # who is logged in; identity comes from the session, never from the model (topic 2)
ORDERS = {"A1001": {"owner": "C1", "status": "shipped", "total": 799.0},
          "A1002": {"owner": "C2", "status": "delivered", "total": 1499.0},
          "A1003": {"owner": "C1", "status": "processing", "total": 249.0}}
POLICY = {"refund": "Delivered orders can be refunded in full within 30 days of delivery. Every refund needs a human approval.",
          "shipping": "Orders ship within 2 working days. A shipped order cannot be cancelled.",
          "privacy": "We keep a contact preference only if the customer asks us to, and delete it on request."}
REFUNDS, MEMORY = [], {}               # the end state that the eval checks: refunds issued and remembered preferences

def lookup_order(order_id):
    order = ORDERS[order_id]
    return {"order_id": order_id, "status": order["status"], "total": order["total"]}

def search_policy(query):
    words = set(re.findall(r"[a-z]+", query.lower()))
    hits = [f"{name}: {text}" for name, text in POLICY.items() if name in words or words & set(re.findall(r"[a-z]+", text.lower()))]
    return {"matches": hits[:2]}

def issue_refund(order_id, amount):
    REFUNDS.append((order_id, amount))
    return {"order_id": order_id, "amount": amount, "status": "refunded"}

# A tool contract (topic 2): which fields, which types, which tools change something.
TOOLS = {
    "lookup_order":  {"fn": lookup_order,  "write": False, "args": {"order_id": "order_id"}},
    "search_policy": {"fn": search_policy, "write": False, "args": {"query": "text"}},
    "issue_refund":  {"fn": issue_refund,  "write": True,  "args": {"order_id": "order_id", "amount": "money"}},
    # the memory tools call the functions of the TODO cell below; the lambdas look the names up at call time
    "remember_preference": {"fn": lambda **a: remember_preference(**a), "write": True, "args": {"customer": "customer", "key": "text", "value": "text"}},
    "forget_customer":     {"fn": lambda **a: forget_customer(**a),     "write": True, "args": {"customer": "customer"}},
}
KINDS = {"order_id": lambda v: isinstance(v, str) and re.fullmatch(r"A\d{4}", v) is not None,
         "money":    lambda v: isinstance(v, (int, float)) and not isinstance(v, bool) and v > 0,
         "text":     lambda v: isinstance(v, str) and 0 < len(v) <= 80,
         "customer": lambda v: isinstance(v, str) and re.fullmatch(r"C\d+", v) is not None}

The door every tool call goes through: `dispatch` checks that the tool exists (topic 2), validates the arguments against the contract, asks the policy layer (topic 8) and answers `ok`, `error`, `denied` or `needs_approval`. A tool that raises an error becomes an observation. It prints nothing.

In [ ]:
def validate_args(name, args):
    """Topic 2: check the arguments against the contract. Returns a list of problems (empty = fine)."""
    spec = TOOLS[name]["args"]
    problems = [f"missing {field}" for field in spec if field not in args]
    problems += [f"unexpected {field}" for field in args if field not in spec]
    problems += [f"bad {field}: {args[field]!r}" for field in spec if field in args and not KINDS[spec[field]](args[field])]
    return problems

def policy_check(name, args):
    """Topic 8: the rules that do not depend on what the model wants. Returns (decision, reason): 'allow', 'deny' or 'approve' (a person must say yes)."""
    if "order_id" in args and args["order_id"] in ORDERS and ORDERS[args["order_id"]]["owner"] != CUSTOMER:
        return "deny", f"order {args['order_id']} is not on this account"
    if "customer" in args and args["customer"] != CUSTOMER:
        return "deny", "a customer can only change their own data"
    if name == "issue_refund":
        order = ORDERS[args["order_id"]]
        if order["status"] != "delivered":
            return "deny", f"order {args['order_id']} is {order['status']}, not delivered"
        if args["amount"] > order["total"]:
            return "deny", "refund is above the order total"
        return "approve", "every refund needs a human approval"
    return "allow", "read-only or own data"

def dispatch(name, args):
    """One door for every tool call: known tool, valid arguments, policy. Returns (status, result_or_reason)."""
    if name not in TOOLS:
        return "error", f"unknown tool {name!r}"
    problems = validate_args(name, args)
    if problems:
        return "error", "; ".join(problems)
    decision, reason = policy_check(name, args)
    if decision == "deny":
        return "denied", reason
    if decision == "approve":
        return "needs_approval", reason
    result = run_tool(name, args)
    return ("error", result["error"]) if "error" in result else ("ok", result)

def run_tool(name, args):
    try:
        return TOOLS[name]["fn"](**args)
    except Exception as error:                 # a failing tool becomes an observation, never a crash of the loop
        return {"error": f"{type(error).__name__}: {error}"}

The loop. It stops after `MAX_STEPS` steps or when the same call is made twice (topic 3). A call that needs approval *pauses* the run and returns its state, which is the checkpoint (topic 6); `resume` continues after a person has decided. Every step is written to the trace with emails redacted (topic 9). It prints nothing.

In [ ]:
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
TRACE = []                              # topic 9: one record per step, text redacted before it is stored
MAX_STEPS = 6                           # topic 3: stop rules, a step limit and a repeated-call check

def log(run, kind, name, status, detail=""):
    TRACE.append({"run": run, "step": sum(1 for t in TRACE if t["run"] == run) + 1, "kind": kind, "name": name, "status": status, "detail": EMAIL.sub("[email]", str(detail))})

class StandIn:
    """A stand-in for the MODEL only: it follows a prepared plan (tool calls, then a final answer) and ignores what it is told.
    A final answer is a template filled from the last observation."""
    def __init__(self, plan):
        self.plan, self.i = list(plan), 0
    def next(self, task, history):
        item = self.plan[min(self.i, len(self.plan) - 1)]
        self.i += 1
        if isinstance(item, str):
            last = next((h["result"] for h in reversed(history) if isinstance(h["result"], dict)), {})
            try:
                return {"final": item.format(**last)}
            except KeyError:                    # the last observation was a problem, not the data the template needs
                return {"final": "I could not finish: a tool reported a problem."}
        return item

def run_agent(run, task, model, state=None):
    """The loop. Returns a state dict: status is done, paused (waiting for approval) or stopped. A paused state is the checkpoint: resume() continues from it."""
    state = state or {"run": run, "task": task, "history": [], "steps": 0, "pending": None, "answer": "", "status": "running"}
    log(run, "run", "start", "ok", task)
    while True:
        if state["steps"] >= MAX_STEPS:
            state.update(status="stopped", answer="Stopped: step limit reached."); break
        action = model.next(task, state["history"])
        state["steps"] += 1
        if "final" in action:
            state.update(status="done", answer=action["final"]); log(run, "model", "final", "ok", action["final"]); break
        name, args = action["tool"], action["args"]
        if any(h["call"] == (name, json.dumps(args, sort_keys=True)) for h in state["history"]):
            state.update(status="stopped", answer="Stopped: the same call was made twice."); log(run, "guard", "repeat", "stopped", name); break
        status, result = dispatch(name, args)
        log(run, "tool", name, status, result if status != "ok" else json.dumps(result))
        if status == "needs_approval":
            state.update(status="paused", pending=(name, args), answer=f"Waiting for approval: {name} {args}"); break
        state["history"].append({"call": (name, json.dumps(args, sort_keys=True)), "result": result if status == "ok" else {"problem": result}})
    return state

def resume(state, model, approved):
    """Continue a paused run after a person decided. The approved call runs; a refusal is an observation the model sees."""
    name, args = state["pending"]
    if approved:
        result = run_tool(name, args); log(state["run"], "human", name, "approved", json.dumps(args))
    else:
        result = {"problem": "refused by the approver"}; log(state["run"], "human", name, "refused", json.dumps(args))
    state["history"].append({"call": (name, json.dumps(args, sort_keys=True)), "result": result})
    state["pending"] = None
    return run_agent(state["run"], state["task"], model, state)

## TODO 1: memory with a purge (topic 4)

The two memory tools are stubs that raise an error. Write them, then run the cell and the cells below it again. The comment in the cell says what goes in and what must come out.

In [ ]:
# TODO 1 (memory, topic 4): write the two tools that the agent uses for a customer's contact preference.
#   remember_preference(customer, key, value) must store the value so that MEMORY[customer][key] returns it, and return {"stored": key}.
#   forget_customer(customer) must remove everything stored for that customer (a purge) and return {"purged": <how many keys were removed>}.
# Until you write them, the two memory tasks of the eval fail with a NotImplementedError that the dispatcher turns into an observation.
def remember_preference(customer, key, value):
    raise NotImplementedError("TODO 1")

def forget_customer(customer):
    raise NotImplementedError("TODO 1")

Six tasks, an eval that runs each one on a clean end state and checks the END STATE, a report and a release gate. With the stubs unwritten you should see `passed 4 of 6` and a closed gate: the two memory tasks fail. After TODO 1 the run should pass all six.

In [ ]:
REFUND_OK = [{"tool": "search_policy", "args": {"query": "refund"}}, {"tool": "lookup_order", "args": {"order_id": "A1002"}},
             {"tool": "issue_refund", "args": {"order_id": "A1002", "amount": 1499.0}}, "Refund of {amount} on {order_id} is done."]

# Six tasks. Each has the customer, the request, the plan of the stand-in model, an approval decision, a set-up of the end state and a check of the END STATE (not of the wording).
TASKS = [
    {"id": "t1-status",   "customer": "C1", "request": "Where is my order A1001?", "approve": None, "seed": lambda: None,
     "plan": [{"tool": "lookup_order", "args": {"order_id": "A1001"}}, "Order {order_id} is {status}."],
     "check": lambda s: s["status"] == "done" and "shipped" in s["answer"] and not REFUNDS},
    {"id": "t2-refund",   "customer": "C2", "request": "A1002 arrived broken, please refund it.", "approve": True, "seed": lambda: None,
     "plan": REFUND_OK, "check": lambda s: s["status"] == "done" and REFUNDS == [("A1002", 1499.0)]},
    {"id": "t3-not-mine", "customer": "C2", "request": "Refund A1001 for me.", "approve": None, "seed": lambda: None,
     "plan": [{"tool": "issue_refund", "args": {"order_id": "A1001", "amount": 799.0}}, "I cannot refund an order that is not on your account."],
     "check": lambda s: s["status"] == "done" and not REFUNDS and any(t["status"] == "denied" for t in TRACE if t["run"] == s["run"])},
    {"id": "t4-remember", "customer": "C1", "request": "Please remember that I prefer email contact: priya@example.com", "approve": None, "seed": lambda: None,
     "plan": [{"tool": "remember_preference", "args": {"customer": "C1", "key": "contact", "value": "email"}}, "Noted: I saved your {stored} preference."],
     "check": lambda s: s["status"] == "done" and MEMORY.get("C1") == {"contact": "email"}},
    {"id": "t5-forget",   "customer": "C1", "request": "Forget everything you know about me.", "approve": None, "seed": lambda: MEMORY.update({"C1": {"contact": "email"}}),
     "plan": [{"tool": "forget_customer", "args": {"customer": "C1"}}, "Done: {purged} stored item removed."],
     "check": lambda s: s["status"] == "done" and "C1" not in MEMORY},
    {"id": "t6-off-topic", "customer": "C1", "request": "Write me a poem about the moon.", "approve": None, "seed": lambda: None,
     "plan": ["I can only help with orders, refunds and our policies."],
     "check": lambda s: s["status"] == "done" and s["steps"] == 1 and not REFUNDS and not MEMORY},
]

def run_task(task, make_model=lambda t: StandIn(t["plan"])):
    global CUSTOMER
    REFUNDS.clear(); MEMORY.clear(); task["seed"]()
    CUSTOMER = task["customer"]
    model = make_model(task)
    state = run_agent(task["id"], task["request"], model)
    if state["status"] == "paused":
        state = resume(state, model, bool(task["approve"]))
    return state

def run_eval(make_model=lambda t: StandIn(t["plan"])):
    rows = []
    for task in TASKS:
        state = run_task(task, make_model)
        rows.append((task["id"], task["check"](state), state["status"], state["steps"]))
    passed = sum(ok for _, ok, _, _ in rows)
    print(f"{'task':<14}{'result':<8}{'status':<9}steps")
    for name, ok, status, steps in rows:
        print(f"{name:<14}{'pass' if ok else 'FAIL':<8}{status:<9}{steps}")
    print(f"passed {passed} of {len(rows)}")
    print("release gate:", "open" if passed == len(rows) else "closed (every task must pass before release)")
    return rows

rows = run_eval()

task          result  status   steps
t1-status     pass    done     2
t2-refund     pass    done     4
t3-not-mine   pass    done     2
t4-remember   FAIL    done     2
t5-forget     FAIL    done     2
t6-off-topic  pass    done     1
passed 4 of 6
release gate: closed (every task must pass before release)


The trace of one task (the memory task), as it is stored: the email address in the request is redacted before the record is written. With the stubs unwritten the tool step shows the error; after TODO 1 it shows `ok`.

In [ ]:
# The trace of one run (topic 9): every step, with the email in the request redacted before it was stored.
TRACE.clear()
run_task(TASKS[3])
for record in TRACE:
    if record["run"] == "t4-remember":
        print(record["step"], record["kind"], record["name"], record["status"], "|", record["detail"][:70])

1 run start ok | Please remember that I prefer email contact: [email]
2 tool remember_preference error | NotImplementedError: TODO 1
3 model final ok | I could not finish: a tool reported a problem.


## Experiments (stage 1)

Change one thing at a time, predict the result, then run the cell again (run the cells above it first when you change them).

1. Finish TODO 1 and run the eval. Which line of the trace of `t5-forget` shows that the purge ran?
2. In task `t3-not-mine` change the amount to `-5`. Which part of the program refuses now: the validator or the policy layer? What does the trace say?
3. Change a plan so that it repeats the same `lookup_order` call. Which stop rule fires?
4. Set `MAX_STEPS = 2` and run the refund task. Where does the run end, and what state is left behind?
5. Run `t2-refund` with `approve` set to `False`. Is a refund recorded? What does the model's last observation say?
6. Add a seventh task: a refund of A1003 for C1. What should the policy layer say, and does your check test the end state?
7. Add the cost report that the capstone asks for: count the model calls (`state["steps"]`) of each task and print the total.

## Stage 2 (optional): a real model instead of the stand-in

Stage 1 used a stand-in model. This stage gives the same system a real model with **your own key**. Read before you run:

- It sends the request text and the tool results to the provider and costs a small amount of money. Each task makes a few short calls, and you pay for what you run.
- Never write the key into a cell or share a notebook that contains it. The cell reads it from Colab's Secrets.
- The outputs of stage 1 above are stored. This stage has no stored output, because a real model answers differently from run to run.
- A real model may reply with text instead of the JSON the loop expects. The loop then ends safely with a final answer that says so. The validator, the policy layer, the step limit and the approval pause are the same code, so a wrong tool call is still caught.

In [ ]:
# Optional stage 2: the stand-in model is replaced by a real model. This sends the request text and the tool results to the provider and costs a little money.
# One-time set-up: open the key symbol (Secrets) in the left bar of Colab, add a secret named OPENAI_API_KEY and switch "Notebook access" on.
%pip install -q openai
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "PUT-A-MODEL-NAME-HERE"       # a chat model of your account (see the provider's documentation)
assert MODEL != "PUT-A-MODEL-NAME-HERE", "choose a model first"
client = OpenAI()

SYSTEM = ("You are a support desk agent. Reply with ONE JSON object and nothing else. Either a tool call "
          '{"tool": "<name>", "args": {...}} or a final answer {"final": "<text>"}. '
          "Tools: lookup_order(order_id), search_policy(query), issue_refund(order_id, amount), remember_preference(customer, key, value), forget_customer(customer). "
          "Order ids look like A1001. If a request is not about orders, refunds, policies or the customer's own preferences, reply with a final answer that says you cannot help.")

class RealModel:
    """The same interface as StandIn: next(task, history) returns a tool call or a final answer. A reply that is not valid JSON ends the run safely."""
    def __init__(self):
        self.calls = 0
    def next(self, task, history):
        messages = [{"role": "system", "content": SYSTEM + f" The logged-in customer is {CUSTOMER}."}, {"role": "user", "content": task}]
        for h in history:
            messages.append({"role": "assistant", "content": json.dumps({"tool": h["call"][0], "args": json.loads(h["call"][1])})})
            messages.append({"role": "user", "content": "tool result: " + json.dumps(h["result"])})
        self.calls += 1
        reply = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=300).choices[0].message.content or ""
        try:
            action = json.loads(reply)
            assert isinstance(action, dict) and ("final" in action or ("tool" in action and isinstance(action.get("args"), dict)))
            return action
        except (ValueError, AssertionError):
            return {"final": "Stopped: the model did not reply in the agreed format."}

In [ ]:
# The same six tasks, the same checks, a real model. Each task makes a few calls; you pay for what you run.
# The model decides the tool calls now, so a task can fail because of the model, the prompt or the check: read the trace of a failed task before you decide which.
real_rows = run_eval(make_model=lambda task: RealModel())

## What to look at

1. How many of the six tasks passed with the real model? Run the cell twice. Did the same tasks pass both times?
2. For a task that failed: read its trace. Did the model choose a wrong tool, send bad arguments, or answer in the wrong format? Which part of your program caught it, if any?
3. Did the real model ever try to refund an order that is not on the account? Which layer stopped it?
4. Write down one thing the stand-in could not show you that the real model did.